# LightKubeGuard — Experiment Pipeline

This notebook reproduces the full LightKubeGuard experimental pipeline step-by-step,
calling the same reusable modules from `src/`.

**Pre-requisite:** Run from the `LightKubeGuard/` project root so that `src/` is on the path.

In [ ]:
import sys, os
# Ensure project root is on sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # use non-interactive backend in notebook if needed
import matplotlib.pyplot as plt
print('Imports OK')

## 1 · Configuration

In [ ]:
from src import config

print(f'Random seed:   {config.RANDOM_SEED}')
print(f'N timesteps:   {config.N_TIMESTEPS}')
print(f'Window size:   {config.WINDOW_SIZE}')
print(f'IF estimators: {config.IF_N_ESTIMATORS}')
print(f'Contamination: {config.IF_CONTAMINATION}')
print(f'Thresholds:    {config.THRESHOLDS}')
print(f'Outputs dir:   {config.OUTPUTS["dir"]}')

## 2 · Generate Synthetic Telemetry

In [ ]:
from src.data_generator import generate_telemetry

df = generate_telemetry()
print(df.shape)
df.head()

In [ ]:
print('Anomaly rate:', df['anomaly_label'].mean().round(4))
print(df.groupby('scenario_name')['anomaly_label'].sum())

In [ ]:
# Quick visualisation of raw signals
fig, axes = plt.subplots(3, 2, figsize=(14, 8))
metrics_to_plot = ['cpu', 'memory', 'net_in', 'net_out', 'latency', 'restarts']
for ax, col in zip(axes.flatten(), metrics_to_plot):
    ax.plot(df['timestep'], df[col], linewidth=0.7)
    # shade anomaly regions
    anom = df['anomaly_label'].values
    in_r = False
    for i, v in enumerate(anom):
        if v == 1 and not in_r:
            s = i; in_r = True
        elif v == 0 and in_r:
            ax.axvspan(s, i, color='red', alpha=0.15)
            in_r = False
    if in_r:
        ax.axvspan(s, len(anom), color='red', alpha=0.15)
    ax.set_title(col)
    ax.set_xlabel('Timestep')
plt.suptitle('Raw Telemetry Signals (red = anomaly regions)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS['dir'], 'raw_signals_overview.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Overview plot saved.')

## 3 · Sliding-Window Feature Extraction

In [ ]:
from src.feature_engineering import extract_features

features_df, scaler, valid_index = extract_features(df)
print(f'Feature matrix: {features_df.shape[0]} rows x {features_df.shape[1]} columns')
print('Features:', list(features_df.columns))
features_df.describe().round(3)

In [ ]:
# Ground-truth labels aligned to valid (post-NaN-drop) index
labels_aligned = df['anomaly_label'].values[valid_index]
print('Aligned anomaly rate:', labels_aligned.mean().round(4))

## 4 · Train Isolation Forest (LightKubeGuard)

In [ ]:
from src.anomaly_model import train_and_predict

if_predictions, if_scores, detector = train_and_predict(features_df, labels_aligned)

print(f'Flagged windows (IF): {if_predictions.sum()} / {len(if_predictions)}')

# Map IF predictions to the full df index
if_predictions_full = np.zeros(len(df), dtype=int)
for j, orig_idx in enumerate(valid_index):
    if_predictions_full[orig_idx] = if_predictions[j]

print(f'Flagged timesteps (IF, full): {if_predictions_full.sum()}')

## 5 · Threshold-Based Baseline

In [ ]:
from src.threshold_baseline import ThresholdDetector

thresh_detector   = ThresholdDetector()
thresh_preds_full = thresh_detector.predict(df)

print(f'Flagged timesteps (Threshold): {thresh_preds_full.sum()}')

## 6 · Evaluation

In [ ]:
from src.evaluation import (
    compute_metrics, evaluate_by_scenario,
    save_results, save_scenario_results, write_paper_summary
)
from src.utils import anomaly_regions_from_labels
from sklearn.metrics import roc_auc_score

y_true_full = df['anomaly_label'].values
regions     = anomaly_regions_from_labels(y_true_full)

result_if = compute_metrics(
    y_true_full, if_predictions_full,
    regions=regions, method_name='LightKubeGuard'
)

# Compute ROC-AUC using aligned labels and scores
try:
    auc = roc_auc_score(labels_aligned, -if_scores)
    result_if['roc_auc'] = round(auc, 4)
    roc_computed = True
    print(f'ROC-AUC (LightKubeGuard): {auc:.4f}')
except Exception as ex:
    roc_computed = False
    print(f'ROC-AUC not computed: {ex}')

result_thresh = compute_metrics(
    y_true_full, thresh_preds_full,
    regions=regions, method_name='Threshold'
)

results = [result_if, result_thresh]
pd.DataFrame(results)[['method','precision','recall','f1_score','fpr','avg_detection_delay','roc_auc']]

In [ ]:
# Per-scenario breakdown
df_scenario = evaluate_by_scenario(df, if_predictions, thresh_preds_full, valid_index)
df_scenario

In [ ]:
# Save to outputs/
save_results(results)
save_scenario_results(df_scenario)
write_paper_summary(results, scenario_computed=True, roc_computed=roc_computed)

## 7 · Figures

In [ ]:
from src.visualization import generate_all_figures

generate_all_figures(df, if_predictions_full, results)
print('All figures saved.')

In [ ]:
# Display the figures inline
from IPython.display import Image, display
for key in ['cpu_plot', 'latency_plot', 'accuracy_plot', 'delay_plot']:
    print(f'--- {key} ---')
    display(Image(filename=config.OUTPUTS[key]))

## 8 · Inspect Paper Summary

In [ ]:
with open(config.OUTPUTS['summary_txt']) as f:
    print(f.read())